# 04 — Embedding congelati + regressione logistica per testata

Il confronto più pulito tra rappresentazione **lessicale** (TF-IDF, `01`) e **semantica** (transformer): lo stesso
classificatore, una regressione logistica, sopra rappresentazioni diverse.

1. Un modello di embedding pre-addestrato e **congelato** (nessun fine-tuning, quindi niente memorizzazione delle
   testate) trasforma ogni articolo nei suoi primi `MAX_TOKEN` token in un vettore, normalizzato.
2. Ogni testata è la **media** dei vettori dei suoi articoli: la stessa granularità dell'etichetta.
3. Regressione logistica (pesi di classe bilanciati) sulle testate di training; **C** scelto in una validazione
   interna a 3 fold per gruppi, sul solo training, con la F1 macro per testata (come `01` con `C_GRIGLIA`).

Ogni articolo riceve la predizione della sua testata: le metriche "per testata" sono quelle che contano, quelle
"per articolo" ripetono la testata.

Si salvano: gli embedding (`Results/2-classificazione/embedding/<NOME>.npy`), per ogni articolo logit e probabilità,
per ogni testata la predizione, il C scelto in ogni fold con le F1 interne, `metriche.json`, `FATTO`
(in `Results/2-classificazione/transformer/<NOME>/`).

In [1]:
import json
import time

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoTokenizer

from comune import CLASSI, RESULTS, SEED, carica, metriche, registratore

/home/fabio.pinelli/mbvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NOME = "e5_large_v2"
MODELLO_EMB = "intfloat/e5-large-v2"
PREFISSO = "query: "        # e5 vuole un prefisso; per bge niente
POOLING = "media"            # "media" dei token (e5) oppure "cls" (bge)
MAX_TOKEN = 512
BATCH = 64
C_GRIGLIA = "0.01,0.1,1,10"
SCHEMA = "fold_gruppo"
PROVA = False

In [3]:
# Parameters
NOME = "e5_large_v2"
MODELLO_EMB = "intfloat/e5-large-v2"
PREFISSO = "query: "
POOLING = "media"


In [4]:
base = RESULTS / ("prova" if PROVA else "")
OUT = base / "transformer" / NOME
OUT.mkdir(parents=True, exist_ok=True)
(base / "embedding").mkdir(parents=True, exist_ok=True)
registra = registratore(base / f"avanzamento_{NOME}.log")
art = carica(PROVA)
json.dump({k: globals()[k] for k in ["NOME", "MODELLO_EMB", "PREFISSO", "POOLING", "MAX_TOKEN", "BATCH", "C_GRIGLIA", "SCHEMA", "PROVA"]},
          open(OUT / "config.json", "w"), indent=1)
registra(f"=== {NOME}: {MODELLO_EMB}, {len(art)} articoli, {art.domain.nunique()} testate ===")

2026-10-18 14:29:43  === e5_large_v2: intfloat/e5-large-v2, 44029 articoli, 304 testate ===


## Gli embedding degli articoli

In [5]:
F_EMB = base / "embedding" / f"{NOME}.npy"
F_IDS = base / "embedding" / f"{NOME}_article_id.npy"
if F_EMB.exists() and F_IDS.exists() and np.array_equal(np.load(F_IDS), art.article_id.to_numpy()):
    E = np.load(F_EMB)
    registra("embedding già calcolati, li carico")
else:
    t0 = time.time()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(MODELLO_EMB)
    mod = AutoModel.from_pretrained(MODELLO_EMB).to(device).eval()
    if device.type == "cuda":
        mod = mod.half()
    testi = [PREFISSO + t for t in art.text.tolist()]
    ordine = np.argsort([len(t) for t in testi])           # lotti di lunghezza simile: meno padding
    E = np.zeros((len(testi), mod.config.hidden_size), dtype=np.float32)
    with torch.no_grad():
        for s in range(0, len(testi), BATCH):
            idx = ordine[s:s + BATCH]
            enc = tok([testi[i] for i in idx], truncation=True, max_length=MAX_TOKEN, padding=True, return_tensors="pt").to(device)
            h = mod(**enc).last_hidden_state.float()
            if POOLING == "cls":
                v = h[:, 0]
            else:
                m = enc.attention_mask.unsqueeze(-1).float()
                v = (h * m).sum(1) / m.sum(1)
            E[idx] = torch.nn.functional.normalize(v, dim=1).cpu().numpy()
    np.save(F_EMB, E); np.save(F_IDS, art.article_id.to_numpy())
    del mod; torch.cuda.empty_cache()
    registra(f"embedding calcolati in {(time.time() - t0) / 60:.1f} min ({E.shape[1]} dimensioni)")

/home/fabio.pinelli/mbvenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


2026-10-18 14:34:20  embedding calcolati in 4.6 min (1024 dimensioni)


## Una riga per testata, e la regressione logistica

In [6]:
testate = art.groupby("domain").agg(label=("label", "first"), gruppo=("gruppo", "first"), fold=(SCHEMA, "first")).reset_index()
pos = art.groupby("domain").indices
X = np.stack([E[pos[d]].mean(0) for d in testate.domain])
X /= np.linalg.norm(X, axis=1, keepdims=True)
griglia = [float(c) for c in C_GRIGLIA.split(",")]


def adatta(Xa, ya, c):
    sc = StandardScaler().fit(Xa)
    return sc, LogisticRegression(C=c, max_iter=5000, class_weight="balanced", random_state=SEED).fit(sc.transform(Xa), ya)


righe_C, logit, proba = [], np.zeros((len(testate), 5)), np.zeros((len(testate), 5))
for fold in sorted(testate.fold.unique()):
    tr, te = np.where(testate.fold != fold)[0], np.where(testate.fold == fold)[0]
    punteggi = {c: [] for c in griglia}
    t_tr = testate.iloc[tr].reset_index(drop=True)
    for itr, iva in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(t_tr, t_tr.label, t_tr.gruppo):
        for c in griglia:
            sc, clf = adatta(X[tr][itr], t_tr.label[itr], c)
            p = clf.predict(sc.transform(X[tr][iva]))
            punteggi[c].append(f1_score(t_tr.label[iva], p, average="macro", labels=range(5), zero_division=0))
    medie = {c: float(np.mean(s)) for c, s in punteggi.items()}
    C = max(medie, key=medie.get)
    sc, clf = adatta(X[tr], testate.label[tr], C)
    logit[np.ix_(te, clf.classes_)] = clf.decision_function(sc.transform(X[te]))
    proba[np.ix_(te, clf.classes_)] = clf.predict_proba(sc.transform(X[te]))
    righe_C.append({"fold": int(fold), "C": C, **{f"F1 interna C={c}": v for c, v in medie.items()}})
    registra(f"fold {fold}: C scelto {C}; F1 interne " + ", ".join(f"{c}: {v:.3f}" for c, v in medie.items()))
pd.DataFrame(righe_C).to_csv(OUT / "scelte_C.csv", index=False)

2026-10-18 14:34:22  fold 1: C scelto 10.0; F1 interne 0.01: 0.271, 0.1: 0.273, 1.0: 0.278, 10.0: 0.320


2026-10-18 14:34:23  fold 2: C scelto 0.1; F1 interne 0.01: 0.303, 0.1: 0.310, 1.0: 0.286, 10.0: 0.277


2026-10-18 14:34:24  fold 3: C scelto 0.01; F1 interne 0.01: 0.345, 0.1: 0.344, 1.0: 0.323, 10.0: 0.337


2026-10-18 14:34:25  fold 4: C scelto 0.01; F1 interne 0.01: 0.317, 0.1: 0.310, 1.0: 0.289, 10.0: 0.294


2026-10-18 14:34:27  fold 5: C scelto 0.01; F1 interne 0.01: 0.349, 0.1: 0.344, 1.0: 0.310, 10.0: 0.276


## Salvataggio e metriche

In [7]:
riga = {d: i for i, d in enumerate(testate.domain)}
i_art = art.domain.map(riga).to_numpy()
pred = pd.DataFrame({"article_id": art.article_id, "domain": art.domain, "fold": art[SCHEMA], "y_true": art.label})
for k in range(5):
    pred[f"logit_l{k + 1}"] = logit[i_art, k]
for k, c in enumerate(CLASSI):
    pred[c] = proba[i_art, k]
pred["y_pred"] = proba[i_art].argmax(1)
pred.to_csv(OUT / "articoli.csv", index=False)
testate.assign(y_pred=proba.argmax(1), **{c: proba[:, k] for k, c in enumerate(CLASSI)}).to_csv(OUT / "testate.csv", index=False)
m = metriche(pred)
json.dump(m, open(OUT / "metriche.json", "w"), indent=1)
(OUT / "FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
registra(f"=== {NOME} FINITO: testata F1 {m['testata_voto']['f1_macro']:.3f}, binario {m['testata_voto']['bin_accuratezza']:.3f}, "
         f"F1 binaria {m['testata_voto']['bin_f1_macro']:.3f} ===")
pd.DataFrame({k: {x: v[x] for x in ["accuratezza", "f1_macro", "bin_accuratezza", "bin_f1_macro", "tolerant_0.5"]} for k, v in m.items()}).T.round(3)

2026-10-18 14:34:28  === e5_large_v2 FINITO: testata F1 0.331, binario 0.727, F1 binaria 0.723 ===


,accuratezza,f1_macro,bin_accuratezza,bin_f1_macro,tolerant_0.5
articolo,0.374,0.319,0.729,0.725,0.582
testata_voto,0.391,0.331,0.727,0.723,0.589
testata_media,0.391,0.331,0.727,0.723,0.589
